# Classwork: The Analyst's Toolkit

**Names:** _(pair)_

After learning about different tools (or a toolkit-- Descriptive statistics, Correlation, Regression/Multiple regression, Chi-square test) of working with data in the previous Mods, in this acitivity you will apply your real skills on the data you have.  
  
  Suppose your company hands you `customers.csv` (40 customers: `age`, `income`, `spending`, `region`, `subscribed`) and a list of questions. You are the analyst. Answer each one with the right tool/technique you decide to apply.

Work in pairs. Write answers in the markdown cells provided.

## Setup
Import the libraries you'll need and load the data.

In [1]:
# import pandas (and numpy, scipy.stats, statsmodels.api as you need them)
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
from scipy.stats import chi2_contingency

# load customers.csv into a DataFrame
df = pd.read_csv("customers.csv")

# take a first look (head, info)
df.head()   # check the first few rows of the DataFrame

# df.info()   # check the structure and data types of the DataFrame

,customer_id,age,income,spending,region,subscribed
0,1,25,61.3,39.1,West,No
1,2,25,54.9,42.1,South,Yes
2,3,51,105.1,59.9,South,No
3,4,39,NaN,49.9,South,No
4,5,43,72.4,48.3,East,No


## Part 0: Know your data, then clean it

This data is dirty on purpose. Find and fix **three** problems before you analyze anything.

In [2]:
# HINT 1: a numeric column has missing values -> decide how to fill them (mean? median? why?) --> income
# prints the "income" column where nan values were filled by taking the median
df["income"] = df['income'].fillna(df['income'].median())
print(df["income"])

# HINT 2: one category label is written in the wrong case -> standardize it --> region
df['region'] = df['region'].str.capitalize()    # we capitalize the word
print(df['region'])

# HINT 3: one value is impossible for its column -> treat it as missing, then fill --> income
df['income_2'] = df['income'].replace(0, np.nan)
print(df['income_2'])

# tip: check .isna().sum() and the unique values of the category column
# checks and numbers for missing values in each column
print(df.isna().sum())


0      61.3
1      54.9
2     105.1
3      84.7
4      72.4
5      84.7
6      92.9
7      53.3
8      71.9
9      70.4
10     92.2
11    106.1
12     81.8
13     70.2
14     96.4
15     67.0
16    105.4
17     84.7
18    108.7
19     84.7
20    108.3
21     83.3
22     81.0
23     75.2
24     77.8
25    -50.0
26     93.5
27     78.5
28    110.1
29     62.6
30     93.5
31    109.4
32     81.1
33     99.2
34     95.2
35     74.7
36    114.5
37    108.4
38     85.9
39     92.7
Name: income, dtype: float64
0      West
1     South
2     South
3     South
4      East
5      East
6      East
7     South
8      East
9      East
10     East
11     East
12    South
13     West
14     East
15     West
16     East
17     West
18     East
19     East
20     East
21     West
22     West
23     East
24     West
25     East
26     East
27     East
28     East
29    South
30     East
31     East
32     West
33    South
34    South
35     West
36     East
37    South
38     West
39     East
Name: regio

**Explain each issue:** what you found and how you handled it.

**First issue is located in 'income' column where some values are 'NaN'.**: I used the "fillna" function to replace all NaN values with a number that is found by getting the median
- the reason I chose the median because it is 'less sensitive' to outliers, which I believe means there are less decimal places that could be used to protray a number being larger/smaller than another.

**Second issue is located in the 'region' column where a cell has a lowercase letter**: I used the captialization method to capitalize the first letter.

**Third issue is locared in the 'income' column where one of the cells has the number -50.**: I used the replace method, which would replace the negative value with a zero.



## Part 1: Pick the tool for each question

For each question: (a) name the data types involved, (b) name the tool, (c) run it, (d) answer in one plain sentence.

### Q1. Do customers who earn more also spend more?

In [3]:
# data types: income = FLOAT/NUMERICAL (continuous value), spending = FLOAT/NUMERICAL (continuous value)
# tool: Pearson Correlation

# Pearson correlation measures the strength and direction of a linear relationship between two continuous variables
correlation, p_value = stats.pearsonr(df['income'], df['spending'])

# prints correlation coefficient
print(f"Correlation: {correlation}")

# prints p-value to determine significance
print(f"P-value: {p_value}")

# hint: two continuous variables moving together


Correlation: 0.5752220076722567
P-value: 0.00010330814996160702


**There is a significant and positive relationship between the customer income and spending.**


### Q2. Is a customer's region related to whether they subscribed?

In [4]:
# data types: region = STRING/CATEGORICAL, subscribed = STRING/CATEGORICAL

# tool: Chi-Square Test of Independence

# prints table showing counts of subs and non-subs by region
chi_table = pd.crosstab(df['region'], df['subscribed'])
print(chi_table)

# test of independence
chi2, p_value, dof, expected = chi2_contingency(chi_table)

print()

# prints results
print(f"Chi-square stats: {chi2}")
print(f"P-value: {p_value}")

# hint: two categorical variables -> build a table with pd.crosstab, then test it


subscribed  No  Yes
region             
East         5   16
South        6    3
West         9    1

Chi-square stats: 13.161904761904761
P-value: 0.0013865281628130974


**There is a significant relationship between a customer's region and whether they subscribed**


### Q3. Can we predict spending from income and age together? How much do they explain? (You can review your Assignment 4 for this)

In [5]:
# data types: predicting a continuous outcome from TWO predictors
    # spending = float/numeric
    # income   = float/numeric
    # age      = floar/numeric

# tool: Multiple Linear Regression

# select predictor variables
X = df[['income', 'age']]

# add intercept
X = sm.add_constant(X)

# select target variable
y = df['spending']

# Fir regression model
model = sm.OLS(y, X).fit()
print(model.summary())

# hint: remember to add a constant; look at R-squared and each predictor's p-value


                            OLS Regression Results                            
Dep. Variable:               spending   R-squared:                       0.505
Model:                            OLS   Adj. R-squared:                  0.478
Method:                 Least Squares   F-statistic:                     18.88
Date:                Mon, 05 Oct 2026   Prob (F-statistic):           2.23e-06
Time:                        22:02:51   Log-Likelihood:                -131.72
No. Observations:                  40   AIC:                             269.4
Df Residuals:                      37   BIC:                             274.5
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         28.1119      4.434      6.340      0.0

**Income and age explain approximately 50.5% of the variation in customer spending (R^2 = 0.505). Both p-values from 'age' and 'income' are both significant predictors of spending, suggesting that customers with higher incomes and older ages tend to spend more.**


### Q4. Which region has the most customers, and what is the average income overall?

In [9]:
# hint: count a categorical column
# take the mean of a numeric column

# prints the count of the customers in each region
region_counts = df['region'].value_counts()

# prints the counts
print(region_counts)

# gets the region with the highest count
most_customers = df['region'].mode()[0]

# calculates the average income
average_income = df['income'].mean()

print()

# prints the results
print(f"Region with most customers: {most_customers}")
print(f"Average income: {average_income:.2f}")


region
East     21
West     10
South     9
Name: count, dtype: int64

Region with most customers: East
Average income: 83.09


**The region with the most customers is East, and the average customer income is $83.09.**


## Part 2: The reveal

Answer these two in a sentence each (no code needed).

**a)** Look back at your four tools (Correlation, Chi-square test, Multiple regression, & Descriptive statistic)). What decided which tool you used each time? (Hint: it was not the wording of the question.)

- continuous and numeric variables are analyzed with correlation or regression
- two categorical variables are analyzed with a chi-square test
- descriptive statistics were used to summarize counts and averages


**b)** In Q1 you found a relationship. Can you tell the company that higher income *causes* higher spending? Why or why not?

**No, becasue correlation shows that income and spending are related, but it does not prove that income causes spending.**


## Bonus

In Q2 you ran a chi-square test. Look at its **expected counts** (not the observed counts). Do they all meet the assumption you learned in the chi-square module? What would you tell the company about how much to trust that result?

In [11]:
# hint: chi2_contingency returns the expected table as its 4th value
# check: are all expected counts >= 5?

# creating the contingency table
table = pd.crosstab(df['region'], df['subscribed'])

# the Chi-Square test
chi2, p_value, dof, expected = chi2_contingency(table)

# print the expected counts
print(expected)

print()

# prints whether all expected counts are at least 5
print((expected >= 5).all())


[[10.5 10.5]
 [ 4.5  4.5]
 [ 5.   5. ]]

False


**No, not all expected counts are at least 5 because one expected count is 4.5, so the Chi-Square result may be less reliable and should be interpreted with caution.**


### Push your code to GitHub